In [1]:
import torch
import time
import os
from google.colab import drive

# 1. Install required packages
print("Installing/verifying required packages...")
!pip install -q ultralytics pyyaml pandas numpy

# 2. Mount Google Drive
drive.mount('/content/drive')

# 3. Define root and project directory paths pointing to the yolo26s_spatial_attention folder
root_dir = "/content/drive/MyDrive/road_crack_detection"
yolo_folder = os.path.join(root_dir, "yolo26s_spatial_attention")

os.makedirs(root_dir, exist_ok=True)
os.makedirs(yolo_folder, exist_ok=True)

print(f"Project Root Directory: {root_dir}")
print(f"YOLO26s-SpatialAttention Output Folder: {yolo_folder}")

# 4. Check CUDA Hardware
print(f"\nPyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {device}")

if device.type == "cuda":
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")

Installing/verifying required packages...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 7.9 MB/s eta 0:00:00
Mounted at /content/drive
Project Root Directory: /content/drive/MyDrive/road_crack_detection
YOLO26s-SpatialAttention Output Folder: /content/drive/MyDrive/road_crack_detection/yolo26s_spatial_attention

PyTorch Version: 2.11.0+cpu
Using Device: cpu


In [2]:
import zipfile
import yaml
import glob

# 1. Define dataset zip path and local extraction path
zip_path = os.path.join(root_dir, "dataa.zip")
extract_to = "/content/dataset"
os.makedirs(extract_to, exist_ok=True)

if os.path.exists(zip_path):
    print(f"Extracting {zip_path} to Colab local runtime...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("Dataset extracted successfully.")
else:
    print(f"Error: Could not find 'dataa.zip' at {root_dir}.")

# 2. Locate data.yaml, configure paths, and assign explicit crack class names
data_yaml_path = None
for r, d, files in os.walk(extract_to):
    if "data.yaml" in files:
        data_yaml_path = os.path.join(r, "data.yaml")
        break

if data_yaml_path and os.path.exists(data_yaml_path):
    dataset_root = os.path.dirname(data_yaml_path)
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)

    # Re-map absolute paths for runtime training and keep validation split intact
    data_config['path'] = dataset_root
    data_config['train'] = os.path.join(dataset_root, "train/images")
    data_config['val'] = os.path.join(dataset_root, "valid/images")

    # Force explicit class name mappings so plots display text names instead of numbers
    data_config['names'] = {
        0: 'Longitudinal Crack',
        1: 'Transverse Crack',
        2: 'Pothole',
        3: 'Alligator Crack'
    }

    with open(data_yaml_path, 'w') as f:
        yaml.dump(data_config, f)

    print(f"Updated data.yaml configured at: {data_yaml_path}")
    print("Configured Class Names:", data_config.get('names'))
else:
    print("Error: data.yaml could not be found.")

Extracting /content/drive/MyDrive/road_crack_detection/dataa.zip to Colab local runtime...
Dataset extracted successfully.
Updated data.yaml configured at: /content/dataset/data.yaml
Configured Class Names: {0: 'Longitudinal Crack', 1: 'Transverse Crack', 2: 'Pothole', 3: 'Alligator Crack'}


In [3]:
import torch.nn as nn
from ultralytics.nn import tasks

# 1. Define Standalone Spatial Attention Module
class SpatialAttention(nn.Module):
    def __init__(self, c1, c2=None, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Compute mean and max pooling across channel dimension
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        y = torch.cat([avg_out, max_out], dim=1)
        y = self.sigmoid(self.conv1(y))
        return x * y

# 2. Inject SpatialAttention into Ultralytics module registry dynamically
tasks.SpatialAttention = SpatialAttention
print("SpatialAttention module defined and registered successfully.")

# 3. Create Custom YOLO26s-SpatialAttention Architecture YAML layout
spatial_yaml_content = """
nc: 4
scales:
  s: [0.33, 0.50, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]          # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]         # 1-P2/4
  - [-1, 3, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]         # 3-P3/8
  - [-1, 6, C3k2, [512, False, 0.25]]  # 4-P3
  - [-1, 1, Conv, [512, 3, 2]]         # 5-P4/16
  - [-1, 6, C3k2, [512, True]]         # 6-P4
  - [-1, 1, Conv, [1024, 3, 2]]        # 7-P5/32
  - [-1, 3, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 2, C2PSA, [1024]]              # 10-P5

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']] # 11
  - [[-1, 6], 1, Concat, [1]]          # 12: cat backbone P4 (index 6)
  - [-1, 3, C3k2, [512, False]]        # 13
  - [-1, 1, SpatialAttention, [512]]   # 14: Spatial Attention applied to P4 neck output

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']] # 15
  - [[-1, 4], 1, Concat, [1]]          # 16: cat backbone P3 (index 4)
  - [-1, 3, C3k2, [256, False]]        # 17
  - [-1, 1, SpatialAttention, [256]]   # 18: Spatial Attention applied to P3 neck output

  - [-1, 1, Conv, [256, 3, 2]]         # 19
  - [[-1, 14], 1, Concat, [1]]         # 20: cat with P4 neck output (index 14)
  - [-1, 3, C3k2, [512, False]]        # 21

  - [-1, 1, Conv, [512, 3, 2]]         # 22
  - [[-1, 10], 1, Concat, [1]]         # 23: cat with backbone P5 (index 10)
  - [-1, 3, C3k2, [1024, True]]        # 24

  - [[18, 21, 24], 1, Detect, [nc]]    # 25: P3, P4, P5 detection heads
"""

config_filename = "/content/yolo26s_spatial_attention.yaml"
with open(config_filename, "w") as f:
    f.write(spatial_yaml_content.strip())

print(f"YOLO26s-SpatialAttention YAML configuration successfully saved to {config_filename}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
SpatialAttention module defined and registered successfully.
YOLO26s-SpatialAttention YAML configuration successfully saved to /content/yolo26s_spatial_attention.yaml


In [4]:
from ultralytics import YOLO

# 1. Instantiate custom YAML model layout containing the Spatial Attention blocks
model = YOLO(config_filename)

# 2. Transfer matching pretrained weights from baseline yolo26s.pt
model.load("yolo26s.pt")

print("YOLO26s-SpatialAttention model successfully initialized and partially weighted via .load('yolo26s.pt').")

Transferred 256/555 items from pretrained weights
YOLO26s-SpatialAttention model successfully initialized and partially weighted via .load('yolo26s.pt').


In [ ]:
import pandas as pd

# Configure training arguments for 640x640 resolution
training_args = {
    "data": data_yaml_path,
    "epochs": 50,
    "imgsz": 640,
    "batch": 8,
    "device": 0 if torch.cuda.is_available() else "cpu",
    "optimizer": "auto",
    "lr0": 0.01,
    "lrf": 0.01,
    "seed": 42,
    "cache": False,
    "save": True,
    "save_period": 10,
    "patience": 10,
    "project": yolo_folder,    # Strictly targets /road_crack_detection/yolo26s_spatial_attention/
    "name": "train_run"
}

print(f"Starting training execution... All outputs will save to: {yolo_folder}/train_run")

start_time = time.time()
results = model.train(**training_args)
training_wall_time = time.time() - start_time

print(f"Training completed in {training_wall_time:.2f} seconds.")

# Export clean training history CSV
run_dir = model.trainer.save_dir
results_csv_path = os.path.join(run_dir, "results.csv")

if os.path.exists(results_csv_path):
    df_results = pd.read_csv(results_csv_path)
    df_results.columns = df_results.columns.str.strip()

    summary_save_path = os.path.join(yolo_folder, "training_validation_metrics_summary.csv")
    df_results.to_csv(summary_save_path, index=False)
    print(f"Metrics exported securely to: {summary_save_path}")

Starting training execution... All outputs will save to: /content/drive/MyDrive/road_crack_detection/yolo26s_spatial_attention/train_run
Ultralytics 8.4.174 🚀 Python-3.13.16 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mod

In [ ]:
import shutil
from pathlib import Path
import numpy as np
import torch
import pandas as pd
import time

# Load the best saved checkpoint from the experiment folder
best_model_path = os.path.join(model.trainer.save_dir, "weights", "best.pt")
best_model = YOLO(best_model_path)
print(f"Loaded best model checkpoint from: {best_model_path}")

# --- 1. EXTRACT & SAVE MODEL PARAMETERS & GFLOPS SEPARATELY ---
total_params = sum(p.numel() for p in best_model.model.parameters())
trainable_params = sum(p.numel() for p in best_model.model.parameters() if p.requires_grad)

gflops = getattr(best_model.model, 'gflops', 0.0)
if gflops == 0.0 and hasattr(best_model, 'info'):
    try:
        gflops = best_model.info(verbose=False)
    except Exception:
        gflops = 0.0

print("\n--- Model Architecture Complexity ---")
print(f"Total Parameters:    {total_params:,}")
print(f"Trainable Parameters:{trainable_params:,}")
print(f"GFLOPs:              {gflops}")

# Save GFLOPs and Parameters to a dedicated separate CSV file
complexity_df = pd.DataFrame({
    "Metric": ["Total_Parameters", "Trainable_Parameters", "GFLOPs"],
    "Value": [total_params, trainable_params, gflops]
})
complexity_save_path = os.path.join(yolo_folder, "model_complexity_summary.csv")
complexity_df.to_csv(complexity_save_path, index=False)
print(f"Model complexity saved separately to: {complexity_save_path}")

# --- 2. RUN VALIDATION EVALUATION ---
val_start_time = time.time()
val_metrics = best_model.val(data=data_yaml_path, split="val", imgsz=640)
val_wall_time = time.time() - val_start_time

# Overall performance metrics
mean_precision = np.mean(val_metrics.box.p)
mean_recall = np.mean(val_metrics.box.r)
map50 = val_metrics.box.map50
map50_95 = val_metrics.box.map

print("\n--- Final Validation Evaluation Scores ---")
print(f"Mean Precision: {mean_precision:.4f}")
print(f"Mean Recall:    {mean_recall:.4f}")
print(f"mAP@50:         {map50:.4f}")
print(f"mAP@50-95:      {map50_95:.4f}")

# --- 3. EXTRACT & SAVE PER-CLASS METRICS SEPARATELY ---
class_names = list(data_config['names'].values())
per_class_results = []

for idx, class_name in enumerate(class_names):
    ap50_class = val_metrics.box.ap50[idx] if hasattr(val_metrics.box, 'ap50') else 0.0
    ap_class = val_metrics.box.ap[idx] if hasattr(val_metrics.box, 'ap') else 0.0
    per_class_results.append({
        "Class": class_name,
        "AP@50": ap50_class,
        "AP@50-95": ap_class
    })
    print(f"Class '{class_name}' -> AP@50: {ap50_class:.4f} | AP@50-95: {ap_class:.4f}")

# Create dedicated folder for per-class metrics
per_class_folder = os.path.join(yolo_folder, "per_class_metrics")
os.makedirs(per_class_folder, exist_ok=True)

per_class_df = pd.DataFrame(per_class_results)
per_class_save_path = os.path.join(per_class_folder, "per_class_ap_summary.csv")
per_class_df.to_csv(per_class_save_path, index=False)
print(f"\nPer-class AP summary saved to separate folder: {per_class_save_path}")

# --- 4. SAVE OVERALL METRICS SUMMARY TABLE ---
val_scores_df = pd.DataFrame({
    "Metric": ["Mean_Precision", "Mean_Recall", "mAP@50", "mAP@50-95", "Inference_Wall_Time_Sec"],
    "Score": [mean_precision, mean_recall, map50, map50_95, val_wall_time]
})
val_scores_save_path = os.path.join(yolo_folder, "validation_evaluation_scores.csv")
val_scores_df.to_csv(val_scores_save_path, index=False)
print(f"Overall validation scores saved to: {val_scores_save_path}")

# --- 5. SAFELY COPY VALIDATION PLOTS ---
local_runs_dir = getattr(val_metrics, 'save_dir', None)
if not local_runs_dir or not os.path.exists(str(local_runs_dir)):
    val_runs = sorted(Path("/content/runs/detect").glob("val*"), key=os.path.getmtime)
    local_runs_dir = val_runs[-1] if val_runs else None

drive_plots_dir = os.path.join(yolo_folder, "validation_evaluation_plots")
if local_runs_dir and os.path.exists(local_runs_dir):
    shutil.copytree(local_runs_dir, drive_plots_dir, dirs_exist_ok=True)
    print(f"Evaluation plots copied to: {drive_plots_dir}")